In [ ]:
# ===== 0. Cấu hình
import os
BRANCH = "htstep_lora"                  # nhánh trên GitHub (yêu cầu ghi htstep-lora; tên thật có gạch dưới)
REPO_URL = "https://github.com/im-xiaoming/Omni-Retriever-Custom.git"
REPO = "/content/Omni-fix"
HF_CLIPS = "GraceFeel/htstep-omni-clips"  # repo HF dataset PRIVATE trong tài khoản của bạn: repo duy nhất được ghi
HTSTEP_COMMIT = "cca1dea"               # github.com/facebookresearch/htstep: annotations.json + taxonomy.csv gốc
HT = "/content/htstep"
WAVE_HOME = "/content/WAVE_HOME"
assert HF_CLIPS and "/" in HF_CLIPS, "điền HF_CLIPS (repo HF dataset private của bạn) rồi chạy lại ô này"
RUN = "lora_bs32"                       # tên lượt train: thư mục trên Drive
DRIVE_RUN = f"/content/drive/MyDrive/omniretriever/htstep_lora/{RUN}"   # checkpoint, vram.json, chunk.json, metrics_*
CLIPS = "/content/htstep_clips"         # clip giải nén (đĩa local, khoảng 44 GB)
META = f"{CLIPS}/meta"
TRAIN, VAL = f"{META}/train_omni_htstep.jsonl", f"{META}/val_omni_htstep.jsonl"
PT_ADAPTER = "/content/adapters/omniretriever-7b"
BENCH_DIR = "/content/drive/MyDrive/Colab Notebooks/code KL/Omni_benchmark_data"
BENCH_ZIP, BENCH_MANIFEST = f"{BENCH_DIR}/benchmark.zip", f"{BENCH_DIR}/omniretriever_bench_final.jsonl"
TRACK = f"{CLIPS}/track_bench1000.jsonl"   # 1 000 record bench cố định (seed 0): chỉ theo dõi quên, không chọn gì
EV = f"{DRIVE_RUN}/eval"
OUTPUT_DIR = f"{DRIVE_RUN}/ckpt"
os.environ.update({"WAVE_PATH": f"{WAVE_HOME}/WAVE-7B", "LORA_CKPT": PT_ADAPTER, "CLIP_ROOT": CLIPS,
                   "BEATS_PATH": f"{WAVE_HOME}/BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt"})

In [ ]:
# ===== 1. Drive, code, thư viện (requirements-htstep.txt), GPU A100
import os, sys, json, glob, re, shutil, signal, subprocess, time, hashlib
from collections import Counter
from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")     # Colab Secrets: HF_TOKEN = token Write của tài khoản sở hữu HF_CLIPS


def sh(cmd, log=None, cwd=None):
    """Chạy lệnh shell, in output (và ghi thêm vào log); nút Stop gửi Ctrl-C cho lệnh để nó dừng gọn."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, start_new_session=True)
    f = open(log, "a", encoding="utf-8") if log else None
    try:
        try:
            for line in p.stdout:
                print(line, end="", flush=True)
                if f:
                    f.write(line)
                    f.flush()
        except KeyboardInterrupt:
            os.killpg(p.pid, signal.SIGINT)
            for line in p.stdout:
                print(line, end="", flush=True)
                if f:
                    f.write(line)
            p.wait()
            raise
        return p.wait()
    finally:
        if f:
            f.close()


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for b in iter(lambda: fh.read(1 << 24), b""):
            h.update(b)
    return h.hexdigest()


if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO], check=True)
sh(f"git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git pull -q --ff-only && git log -1 --format='commit %H %s'")
assert sh("pip install -q -r requirements-htstep.txt") == 0
sh("""python -c "import sys, platform, decord, av, transformers, peft, huggingface_hub, torch; print('python', platform.python_version(), '| decord', decord.__version__, '| av', av.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__, '| hub', huggingface_hub.__version__, '| torch', torch.__version__)" """)
sh("ffmpeg -version | head -1")
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2 ** 30
print(f"nproc {os.cpu_count()} | RAM {ram:.1f} GB | đĩa trống /content {shutil.disk_usage('/content').free / 2 ** 30:.0f} GB")
GPU = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip() if shutil.which("nvidia-smi") else ""
print("GPU:", GPU or "không có")
if "A100" not in GPU:
    raise SystemExit(f"Cần GPU A100 (runtime hiện tại: {GPU or 'không có GPU'}): Runtime -> Change runtime type -> A100")
for d in (DRIVE_RUN, EV, OUTPUT_DIR):
    os.makedirs(d, exist_ok=True)
ERRORS = []                                           # lỗi gặp phải, in ở ô 8

In [ ]:
# ===== 2. WAVE-7B + BEATs, adapter gốc của tác giả (sha256 so với YunzeLiu/OmniRetriever-7B), vá RoPE
if not os.path.exists(f"{WAVE_HOME}/WAVE-7B/config.json"):
    sh(f"hf download nguyenminh04/omni-model --local-dir {WAVE_HOME}")
from huggingface_hub import HfApi
author = HfApi().get_paths_info("YunzeLiu/OmniRetriever-7B", ["adapter_model.safetensors"], expand=True)[0].lfs.sha256
ad = f"{PT_ADAPTER}/adapter_model.safetensors"
if not os.path.exists(ad):
    sh(f'hf download nguyenminh04/omni-adapters --include "omniretriever-7b/*" --local-dir {os.path.dirname(PT_ADAPTER)}')
if sha256(ad) != author:
    print("adapter của nguyenminh04/omni-adapters khác bản tác giả -> dùng YunzeLiu/OmniRetriever-7B")
    shutil.rmtree(PT_ADAPTER)
    sh(f"hf download YunzeLiu/OmniRetriever-7B adapter_model.safetensors adapter_config.json --local-dir {PT_ADAPTER}")
ok = sha256(ad) == author
print("CHECK adapter_is_author", "OK" if ok else "FAIL", author)
assert ok, "adapter khác bản của tác giả: dừng"
assert sh(f"python scripts/patch_wave_rope.py {WAVE_HOME}/WAVE-7B") == 0

In [ ]:
# ===== 3. Clip + manifest từ repo HF clip (kiểm đĩa trống trước, giải nén, kiểm sha256), benchmark, taxonomy
rc = sh(f"python scripts/precut_htstep.py fetch --hf-repo {HF_CLIPS} --out {CLIPS}", log=f"{DRIVE_RUN}/fetch.log")
assert rc == 0, "fetch lỗi (thiếu đĩa / sha256 sai): xem fetch.log"
for p in (TRAIN, VAL):
    n = sum(1 for _ in open(p, encoding="utf-8"))
    print(f"{os.path.basename(p)}: {n} item, sha256 {sha256(p)}")
if not os.path.isdir("/content/benchmark/videos"):
    shutil.copy(BENCH_ZIP, "/content/benchmark.zip")
    sh("7z x /content/benchmark.zip -o/content/benchmark -y > /dev/null && "
       "(mv /content/benchmark/benchmark/* /content/benchmark/ 2>/dev/null; rm -rf /content/benchmark/benchmark; true)",
       cwd="/content")
if not os.path.exists(TRACK):
    assert sh(f"python scripts/eval_bench.py --make-track {TRACK} --manifest '{BENCH_MANIFEST}' --media-root /content") == 0
if not os.path.isdir(HT):
    sh(f"git clone -q https://github.com/facebookresearch/htstep.git {HT}")
sh(f"git checkout -q {HTSTEP_COMMIT}", cwd=HT)
print(f"đĩa trống /content {shutil.disk_usage('/content').free / 2 ** 30:.0f} GB")

In [ ]:
# ===== 4. Đo VRAM, chọn chunk K của GradCache (K = 1, 2, 4, 8; 3 bước batch 32 mỗi K) -> chunk.json, protocol.
# Lần chạy sau: đọc chunk.json, bỏ qua bước đo.
CHUNK, PROTO = f"{DRIVE_RUN}/chunk.json", f"{DRIVE_RUN}/protocol.json"
if not os.path.exists(CHUNK):
    rc = sh(f"python scripts/probe_htstep.py --train-manifest {TRAIN} --val-manifest {VAL} --clip-root {CLIPS} "
            f"--out {DRIVE_RUN}", log=f"{DRIVE_RUN}/probe.log")
    assert rc == 0, "probe lỗi / không K nào vừa 90 % VRAM: gửi probe.log + vram_probe.json cho agent"
else:
    P = json.load(open(f"{DRIVE_RUN}/vram_probe.json"))
    print(f"VRAM (GB): sau khi load {P['after_load']} | eval batch 1 {P['eval_batch1']}")
    for k, r in P["train"].items():
        print(f"K={k}: {r['status']} alloc {r['max_allocated_gb']} reserved {r['max_reserved_gb']} {r.get('sec_per_step')} s/bước")
C = json.load(open(CHUNK))
print(f"K = {C['grad_cache_chunk']} | eval mỗi {C['eval_every_frac']} epoch | 1 epoch ~{C['epoch_hours']} h | "
      f"1 lượt val ~{C['val_pass_hours']} h | cả lượt tối đa ~{C['run_hours_max']} h ~ {C['colab_sessions']} phiên Colab")
if not os.path.exists(PROTO):
    assert sh(f"python scripts/make_protocol.py write --grad-cache-chunk {C['grad_cache_chunk']} "
              f"--eval-every-frac {C['eval_every_frac']} --data-dir {META} --out {PROTO}") == 0

In [ ]:
# ===== 5. Mốc 0-shot (pt): val_seen, OmniRetriever-Bench 12 hướng, vector bước cho UniAV. Đã có thì bỏ qua
if not os.path.exists(f"{EV}/metrics_htstep_pt.json"):
    if sh(f"python scripts/eval_htstep.py --manifest {VAL} --adapter {PT_ADAPTER} --tag pt --out {EV}",
          log=f"{EV}/eval_htstep_pt.log"):
        ERRORS.append(f"eval val_seen pt lỗi, xem {EV}/eval_htstep_pt.log")
if not os.path.exists(f"{EV}/bench/bench_pt.json"):
    if sh(f"python scripts/eval_bench.py --manifest '{BENCH_MANIFEST}' --media-root /content --adapter {PT_ADAPTER} "
          f"--tag pt --out {EV}/bench", log=f"{EV}/bench_pt.log"):
        ERRORS.append(f"bench pt lỗi, xem {EV}/bench_pt.log")
if not os.path.exists(f"{DRIVE_RUN}/omni_step_emb_pt.npz"):
    if sh(f"python scripts/export_step_emb.py --adapter {PT_ADAPTER} --taxonomy {HT}/data/taxonomy.csv --tag pt "
          f"--out {DRIVE_RUN}/omni_step_emb_pt.npz", log=f"{EV}/export_pt.log"):
        ERRORS.append(f"export_step_emb pt lỗi, xem {EV}/export_pt.log")

In [ ]:
# ===== 6. Train (training/train.sh, cấu hình mục 2, GradCache K từ chunk.json). Tự resume từ checkpoint cuối.
# Phiên bị ngắt: mở lại notebook, chạy ô 0-3 rồi ô này (ô 4, 5 tự bỏ qua) cho đến khi in "train xong".
def train_state():
    """(xong?, lý do) từ checkpoint cuối và eval_state.json."""
    es = json.load(open(f"{OUTPUT_DIR}/eval_state.json")) if os.path.exists(f"{OUTPUT_DIR}/eval_state.json") else {}
    if es.get("bad_evals", 0) >= 3:
        return True, f"dừng sớm: 3 lần eval không tăng, best R@1 {es['best']} ở bước {es['best_step']}"
    ck = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
    if ck:
        ts = json.load(open(f"{ck[-1]}/trainer_state.json"))
        if ts["global_step"] >= ts["max_steps"]:
            return True, f"đủ 2 epoch ({ts['global_step']} bước)"
    return False, ""

C = json.load(open(f"{DRIVE_RUN}/chunk.json"))
assert sh(f"python scripts/make_protocol.py check --data-dir {META} --protocol {DRIVE_RUN}/protocol.json") == 0, \
    "CHECK protocol FAIL: dữ liệu khác lúc ghi protocol"
done, why = train_state()
if done:
    print("train xong:", why)
else:
    resume = bool(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
    os.environ.update({
        "DATA_PATH": TRAIN, "VIDEO_ROOT": CLIPS, "OUTPUT_DIR": OUTPUT_DIR,
        "LORA_ONLY": "True", "LORA_INIT_ONLY": "False" if resume else "True",   # False: resume từ checkpoint cuối
        "USE_TUPLE_INFONCE": "False", "GRADIENT_CHECKPOINTING": "True", "USE_DEEPSPEED": "False",
        "BATCH_SIZE": "32", "GRAD_ACCUM": "1", "EPOCHS": "2", "LR": "1e-4", "NUM_GPUS": "1", "TRAIN_SEED": "42",
        "ATTN_IMPL": "sdpa"})
    extra = (f"--adam_beta1 0.9 --adam_beta2 0.95 --seed 42 --grad_cache_chunk {C['grad_cache_chunk']} "
             f"--mask_same_text True --eval_manifest {VAL} --eval_every_frac {C['eval_every_frac']} "
             f"--early_stop_patience 3 --track_manifest {TRACK} --save_steps 200 --save_total_limit 1 "
             f"--run_name {RUN}")
    TRAIN_LOG = f"{DRIVE_RUN}/train_{time.strftime('%Y%m%d_%H%M')}.log"
    print("resume" if resume else "lượt đầu (adapter pt)", "->", TRAIN_LOG)
    rc = sh(f"bash training/train.sh {extra}", log=TRAIN_LOG)
    log = open(TRAIN_LOG, errors="replace").read()
    block = log.split("Trainable parameters by group:")[-1].split("=" * 70)[0] if "by group" in log else ""
    groups = [m.groups() for m in map(re.compile(r"^\s+(\S+)\s+([\d,]+)\s*$").match, block.splitlines()) if m]
    print("CHECK lora_only", "OK" if groups == [("LoRA", "6,881,280")] else "FAIL", groups)
    if groups != [("LoRA", "6,881,280")]:
        ERRORS.append(f"train: tham số được train khác 'LoRA 6,881,280': {groups}")
    if rc:
        ERRORS.append(f"train exit {rc}: {log[log.rfind('Traceback'):][-800:] if 'Traceback' in log else log[-800:]}")
    for pat, what in ((r"\[OOM\] Skipping", "batch bị bỏ vì OOM"), (r"\[(?:ERROR|TIMEOUT)\] Sample", "mẫu lỗi"),
                      (r"\[eval\] FAIL", "lần eval có item val không nạp được")):
        n = len(re.findall(pat, log))
        if n:
            ERRORS.append(f"train: {n} {what}, xem {TRAIN_LOG}")
    done, why = train_state()
    print("train xong:" if done else "CHƯA xong (chạy lại ô này):", why)
V = json.load(open(f"{OUTPUT_DIR}/vram.json")) if os.path.exists(f"{OUTPUT_DIR}/vram.json") else {}
print("VRAM:", {k: v for k, v in V.items() if k not in ("log", "step_seconds")})
M = json.load(open(f"{OUTPUT_DIR}/metrics.json")) if os.path.exists(f"{OUTPUT_DIR}/metrics.json") else {}
for h in M.get("history", []):
    print(f"eval bước {h['step']:>5} (epoch {h['epoch']}): t2v R@1 {h['t2v']['R@1']:.2f} (multi {h['t2v']['R@1_multi']:.2f})"
          f" v2t R@1 {h['v2t']['R@1']:.2f} | {h['seconds']:.0f} s")
for t in M.get("track", []):
    print(f"theo dõi bench bước {t['step']}: t2av R@1 {t['t2av']['R@1']:.2f} av2t R@1 {t['av2t']['R@1']:.2f}")

In [ ]:
# ===== 7. Sau khi train xong: checkpoint best trên val_seen + bench 12 hướng, vector bước cho UniAV
done, why = train_state()
assert done, "train chưa xong: chạy lại ô 6"
BEST = f"{OUTPUT_DIR}/best"
if not os.path.isdir(BEST):
    ERRORS.append("không có checkpoint best: không lần eval nào vượt pt; kết quả 'best' = pt")
else:
    if not os.path.exists(f"{EV}/metrics_htstep_best.json"):
        if sh(f"python scripts/eval_htstep.py --manifest {VAL} --lora {BEST} --pt-adapter {PT_ADAPTER} --tag best "
              f"--out {EV}", log=f"{EV}/eval_htstep_best.log"):
            ERRORS.append(f"eval val_seen best lỗi, xem {EV}/eval_htstep_best.log")
    if not os.path.exists(f"{EV}/bench/bench_best.json"):
        if sh(f"python scripts/eval_bench.py --manifest '{BENCH_MANIFEST}' --media-root /content --lora {BEST} "
              f"--pt-adapter {PT_ADAPTER} --tag best --out {EV}/bench", log=f"{EV}/bench_best.log"):
            ERRORS.append(f"bench best lỗi, xem {EV}/bench_best.log")
    if not os.path.exists(f"{DRIVE_RUN}/omni_step_emb_best.npz"):
        if sh(f"python scripts/export_step_emb.py --lora {BEST} --pt-adapter {PT_ADAPTER} --taxonomy {HT}/data/taxonomy.csv "
              f"--tag best --out {DRIVE_RUN}/omni_step_emb_best.npz", log=f"{EV}/export_best.log"):
            ERRORS.append(f"export_step_emb best lỗi, xem {EV}/export_best.log")

In [ ]:
# ===== 8. Tóm tắt: pt / best, đường dẫn kết quả
def load(p):
    return json.load(open(p)) if os.path.exists(p) else None

H = {t: load(f"{EV}/metrics_htstep_{t}.json") for t in ("pt", "best")}
B = {t: load(f"{EV}/bench/bench_{t}.json") for t in ("pt", "best")}
M = load(f"{OUTPUT_DIR}/metrics.json") or {}
print(f"HT-Step val_seen (N = {(H['pt'] or {}).get('n')}), giao thức bài báo (đường chéo):")
for d, name in (("t2v", "T->AV"), ("v2t", "AV->T")):
    for k in ("R@1", "R@5", "R@10", "NDCG@10", "R@1_multi"):
        a, b = ((H[t] or {}).get(d, {}).get(k) for t in ("pt", "best"))
        print(f"  {name} {k:9s} pt {a if a is None else round(a, 2)!s:>7} | best {b if b is None else round(b, 2)!s:>7}")
if H["pt"] and M.get("history"):
    print(f"  (kiểm: pt trong lúc train, bước 0: t2v R@1 {M['history'][0]['t2v']['R@1']:.2f})")
print(f"OmniRetriever-Bench (N = {(B['pt'] or {}).get('n_records')}, bài báo 3782: không so thẳng nếu khác), R@1:")
for d in (B["pt"] or {}).get("metrics", {}):
    a = B["pt"]["metrics"][d]["R@1"]
    b = (B["best"] or {}).get("metrics", {}).get(d, {}).get("R@1")
    print(f"  {d:6s} pt {100 * a:6.2f} | best {'-' if b is None else f'{100 * b:6.2f}'}")
SUMMARY = {"run": RUN, "chunk": load(f"{DRIVE_RUN}/chunk.json"), "best_step": M.get("best_step"), "val_seen": H,
           "bench": {t: (b or {}).get("metrics") for t, b in B.items()}, "errors": ERRORS,
           "files": sorted(glob.glob(f"{DRIVE_RUN}/*.json") + glob.glob(f"{DRIVE_RUN}/*.npz") + glob.glob(f"{EV}/*.json")
                           + glob.glob(f"{EV}/bench/*.json"))}
json.dump(SUMMARY, open(f"{DRIVE_RUN}/summary.json", "w"), indent=1)
print("lỗi:", len(ERRORS))
for e in ERRORS:
    print(" -", e)
print("->", DRIVE_RUN, "(summary.json, metrics.json, vram.json, chunk.json, protocol.json, eval/, omni_step_emb_*.npz)")